In [1]:
import pandas as pd 
import numpy as numpy
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [2]:
rawDf=pd.read_csv("../Data/train.csv")
print(rawDf.columns)
rawDf.drop(columns=['qid1','qid2','id'],inplace=True)
rawDf.sample(4)

Index(['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate'], dtype='object')


,question1,question2,is_duplicate
334062,Which programming language is in demand?,Which programming language will be in demand?,1
219872,Which is the best laptop to buy under 30k in I...,"Which is the best laptop under INR 30,000?",1
378832,How do I find a GOOD therapist that specialize...,"How do I find a good therapist in Santa Clara,...",1
141226,Is there an alliance of Egypt's opposition par...,Is there an alliance of Egypt's left-wing oppo...,1


PREPROCESSING

In [3]:
import re
import nltk
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# --- Initialization ---
# Downloading resources required for Lemmatization and Stopword filtering
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('omw-1.4', quiet=True)

def preprocess_complete(q):
    # 1. Basic Cleaning
    # Convert to string (to handle NaNs), lowercase, and strip whitespace
    q = str(q).lower().strip()
    
    # 2. HTML Tag Removal
    # Questions scraped from the web often contain <html> or &nbsp; tags
    q = BeautifulSoup(q, "html.parser").get_text()
    
    # 3. Currency & Special Character Substitution
    # Replaces symbols with words so the model understands the context (e.g., a money-related question)
    q = q.replace('%', ' percent ')
    q = q.replace('$', ' dollar ')
    q = q.replace('₹', ' rupee ')
    q = q.replace('€', ' euro ')
    q = q.replace('@', ' at ')
    q = q.replace('[math]', '') # Remove specific dataset noise
    
    # 4. Number Normalization
    # Standardizes "1000000" and "1,000,000" into "1m" to help match similar quantities
    q = q.replace(',', '') 
    q = re.sub(r'(\d+)000000000', r'\1b', q)
    q = re.sub(r'(\d+)000000', r'\1m', q)
    q = re.sub(r'(\d+)000', r'\1k', q)
    
    # 5. Full Contraction Expansion
    # Standardizes "won't" to "will not" so the model sees the individual tokens
    contractions = { 
        "ain't": "am not", "aren't": "are not", "can't": "can not", "can't've": "can not have",
        "'cause": "because", "could've": "could have", "couldn't": "could not", "didn't": "did not",
        "doesn't": "does not", "don't": "do not", "hadn't": "had not", "hasn't": "has not",
        "haven't": "have not", "he'd": "he would", "he'll": "he will", "he's": "he is",
        "how'd": "how did", "how'll": "how will", "how's": "how is", "i'd": "i would",
        "i'll": "i will", "i'm": "i am", "i've": "i have", "isn't": "is not", "it'd": "it would",
        "it'll": "it will", "it's": "it is", "let's": "let us", "must've": "must have",
        "mustn't": "must not", "shan't": "shall not", "she'd": "she would", "she'll": "she will",
        "she's": "she is", "should've": "should have", "shouldn't": "should not", "that's": "that is",
        "there's": "there is", "they'd": "they would", "they'll": "they will", "they're": "they are",
        "they've": "they have", "wasn't": "was not", "we'd": "we would", "we'll": "we will",
        "we're": "we are", "we've": "we have", "weren't": "were not", "what's": "what is",
        "where's": "where is", "who's": "who is", "won't": "will not", "would've": "would have",
        "wouldn't": "would not", "you'd": "you would", "you'll": "you will", "you're": "you are",
        "you've": "you have"
    }
    
    q_decontracted = [contractions[word] if word in contractions else word for word in q.split()]
    q = ' '.join(q_decontracted)
    
    # 6. Punctuation & Non-Alphanumeric Removal
    # Keeps only letters and numbers; replaces punctuation with a space to prevent word merging
    q = re.sub(r'[^a-zA-Z0-9\s]', ' ', q)
    
    # 7. Stopwords & Lemmatization
    # We remove common words (the, a, an) but KEEP interrogatives (how, why, what) 
    # and negations (not, no) because they define the intent of a question.
    stop_words = set(stopwords.words('english'))
    keep_list = {'how', 'why', 'what', 'when', 'where', 'not', 'no', 'nor', 'neither', 'none'}
    final_stop_words = stop_words - keep_list
    
    lemmatizer = WordNetLemmatizer()
    
    # Lemmatization: Reduces words to their root (e.g., 'rocks' -> 'rock', 'better' -> 'good')
    # This ensures "How to play guitar" matches "Best way of playing guitar"
    words = q.split()
    q = " ".join([lemmatizer.lemmatize(word) for word in words if word not in final_stop_words])
    
    return q.strip()

# --- Example Usage ---
question1 = "What's the best way to lose 10,000 grams?"
print(preprocess_complete(question1))

what best way lose 10k gram


In [4]:
rawDf['question1'] = rawDf['question1'].apply(preprocess_complete)
rawDf['question2'] = rawDf['question2'].apply(preprocess_complete)
print(rawDf.head())

                                           question1  \
0     what step step guide invest share market india   
1               what story kohinoor koh noor diamond   
2   how increase speed internet connection using vpn   
3                      why mentally lonely how solve   
4  one dissolve water quikly sugar salt methane c...   

                                           question2  is_duplicate  
0           what step step guide invest share market             0  
1  what would happen indian government stole kohi...             0  
2           how internet speed increased hacking dns             0  
3        find remainder when 23 24 math divided 2423             0  
4                      fish would survive salt water             0  


BASIC FEATURES

In [5]:
rawDf['q1_len'] = rawDf['question1'].str.len() 
rawDf['q2_len'] = rawDf['question2'].str.len()
rawDf['q1_num_words'] = rawDf['question1'].apply(lambda row: len(row.split(" ")))
rawDf['q2_num_words'] = rawDf['question2'].apply(lambda row: len(row.split(" ")))

In [6]:
def common_words(row):
    w1 = set(map(lambda word: word.lower().strip(), row['question1'].split(" ")))
    w2 = set(map(lambda word: word.lower().strip(), row['question2'].split(" ")))    
    return len(w1 & w2)
rawDf['word_common'] = rawDf.apply(common_words, axis=1)

def total_words(row):
    w1 = set(map(lambda word: word.lower().strip(), row['question1'].split(" ")))
    w2 = set(map(lambda word: word.lower().strip(), row['question2'].split(" ")))    
    return (len(w1) + len(w2))

rawDf['word_total'] = rawDf.apply(total_words, axis=1)
rawDf['word_share'] = round(rawDf['word_common']/rawDf['word_total'],2)

In [7]:
def fetch_token_features(row):
    
    q1 = row['question1']
    q2 = row['question2']
    
    SAFE_DIV = 0.0001 

    STOP_WORDS = stopwords.words("english")
    
    token_features = [0.0]*8
    
    # Converting the Sentence into Tokens: 
    q1_tokens = q1.split()
    q2_tokens = q2.split()
    
    if len(q1_tokens) == 0 or len(q2_tokens) == 0:
        return token_features

    # Get the non-stopwords in Questions
    q1_words = set([word for word in q1_tokens if word not in STOP_WORDS])
    q2_words = set([word for word in q2_tokens if word not in STOP_WORDS])

    #Get the stopwords in Questions
    q1_stops = set([word for word in q1_tokens if word in STOP_WORDS])
    q2_stops = set([word for word in q2_tokens if word in STOP_WORDS])
    
    # Get the common non-stopwords from Question pair
    common_word_count = len(q1_words.intersection(q2_words))
    
    # Get the common stopwords from Question pair
    common_stop_count = len(q1_stops.intersection(q2_stops))
    
    # Get the common Tokens from Question pair
    common_token_count = len(set(q1_tokens).intersection(set(q2_tokens)))
    
    
    token_features[0] = common_word_count / (min(len(q1_words), len(q2_words)) + SAFE_DIV)
    token_features[1] = common_word_count / (max(len(q1_words), len(q2_words)) + SAFE_DIV)
    token_features[2] = common_stop_count / (min(len(q1_stops), len(q2_stops)) + SAFE_DIV)
    token_features[3] = common_stop_count / (max(len(q1_stops), len(q2_stops)) + SAFE_DIV)
    token_features[4] = common_token_count / (min(len(q1_tokens), len(q2_tokens)) + SAFE_DIV)
    token_features[5] = common_token_count / (max(len(q1_tokens), len(q2_tokens)) + SAFE_DIV)
    
    # Last word of both question is same or not
    token_features[6] = int(q1_tokens[-1] == q2_tokens[-1])

    # First word of both question is same or not
    token_features[7] = int(q1_tokens[0] == q2_tokens[0])
    
    return token_features

In [8]:
token_features = rawDf.apply(fetch_token_features, axis=1)

rawDf["cwc_min"]       = list(map(lambda x: x[0], token_features))
rawDf["cwc_max"]       = list(map(lambda x: x[1], token_features))
rawDf["csc_min"]       = list(map(lambda x: x[2], token_features))
rawDf["csc_max"]       = list(map(lambda x: x[3], token_features))
rawDf["ctc_min"]       = list(map(lambda x: x[4], token_features))
rawDf["ctc_max"]       = list(map(lambda x: x[5], token_features))
rawDf["last_word_eq"]  = list(map(lambda x: x[6], token_features))
rawDf["first_word_eq"] = list(map(lambda x: x[7], token_features))

DISTANCE BASED FEATURES

In [9]:
%pip install distance
import distance

def fetch_length_features(row):
    
    q1 = row['question1']
    q2 = row['question2']
    
    length_features = [0.0]*3
    
    # Converting the Sentence into Tokens: 
    q1_tokens = q1.split()
    q2_tokens = q2.split()
    
    if len(q1_tokens) == 0 or len(q2_tokens) == 0:
        return length_features
    
    # Absolute length features
    length_features[0] = abs(len(q1_tokens) - len(q2_tokens))
    
    #Average Token Length of both Questions
    length_features[1] = (len(q1_tokens) + len(q2_tokens))/2
    
    strs = list(distance.lcsubstrings(q1, q2))
    if strs:
        length_features[2] = len(strs[0]) / (min(len(q1), len(q2)) + 1)
    else:
        length_features[2] = 0.0
    
    return length_features

length_features = rawDf.apply(fetch_length_features, axis=1)

rawDf['abs_len_diff'] = list(map(lambda x: x[0], length_features))
rawDf['mean_len'] = list(map(lambda x: x[1], length_features))
rawDf['longest_substr_ratio'] = list(map(lambda x: x[2], length_features))

Note: you may need to restart the kernel to use updated packages.


In [10]:
# Fuzzy Features
%pip install fuzzywuzzy
from fuzzywuzzy import fuzz

def fetch_fuzzy_features(row):
    
    q1 = row['question1']
    q2 = row['question2']
    
    fuzzy_features = [0.0]*4
    
    # fuzz_ratio
    fuzzy_features[0] = fuzz.QRatio(q1, q2)

    # fuzz_partial_ratio
    fuzzy_features[1] = fuzz.partial_ratio(q1, q2)

    # token_sort_ratio
    fuzzy_features[2] = fuzz.token_sort_ratio(q1, q2)

    # token_set_ratio
    fuzzy_features[3] = fuzz.token_set_ratio(q1, q2)

    return fuzzy_features


fuzzy_features = rawDf.apply(fetch_fuzzy_features, axis=1)

# Creating new feature columns for fuzzy features
rawDf['fuzz_ratio'] = list(map(lambda x: x[0], fuzzy_features))
rawDf['fuzz_partial_ratio'] = list(map(lambda x: x[1], fuzzy_features))
rawDf['token_sort_ratio'] = list(map(lambda x: x[2], fuzzy_features))
rawDf['token_set_ratio'] = list(map(lambda x: x[3], fuzzy_features))

Note: you may need to restart the kernel to use updated packages.


C:\Users\HP\AppData\Roaming\Python\Python314\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [12]:
import pickle
import pandas as pd

all_qs = pd.Series(rawDf['question1'].tolist() + rawDf['question2'].tolist()).astype(str)
freq_map_global = all_qs.value_counts().to_dict()

pickle.dump(freq_map_global, open("../Model/freq_map.pkl", 'wb') )

print(f"Success! Map saved with {len(freq_map_global)} unique questions.")


Success! Map saved with 518189 unique questions.


In [13]:
import pandas as pd
import spacy

# 1. Load model with only what we need
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

def get_nlp_ratios(df):
    # Process all questions in large batches
    q1_docs = list(nlp.pipe(df['question1'].astype(str), batch_size=1000))
    q2_docs = list(nlp.pipe(df['question2'].astype(str), batch_size=1000))
    
    noun_ratios = []
    verb_ratios = []
    
    for doc1, doc2 in zip(q1_docs, q2_docs):
        # Extract sets of nouns and verbs
        n1 = {token.text.lower() for token in doc1 if token.pos_ == 'NOUN'}
        n2 = {token.text.lower() for token in doc2 if token.pos_ == 'NOUN'}
        v1 = {token.text.lower() for token in doc1 if token.pos_ == 'VERB'}
        v2 = {token.text.lower() for token in doc2 if token.pos_ == 'VERB'}
        
        # Calculate Jaccard-style ratios
        n_union = len(n1 | n2)
        v_union = len(v1 | v2)
        
        noun_ratios.append(len(n1 & n2) / n_union if n_union > 0 else 0)
        verb_ratios.append(len(v1 & v2) / v_union if v_union > 0 else 0)
        
    return noun_ratios, verb_ratios

# Run this - it should be much faster
rawDf['noun_ratio'], rawDf['verb_ratio'] = get_nlp_ratios(rawDf)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import paired_cosine_distances

model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all questions in two big batches
# This is MUCH faster than row-by-row apply()
q1_embeddings = model.encode(rawDf['question1'].tolist(), show_progress_bar=True)
q2_embeddings = model.encode(rawDf['question2'].tolist(), show_progress_bar=True)

# paired_cosine_distances gives the distance (0 to 2)
# Similarity = 1 - distancec:\Users\HP\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21
rawDf['cosine_sim'] = 1 - paired_cosine_distances(q1_embeddings, q2_embeddings)

C:\Users\HP\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5039.70it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches:  56%|█████▌    | 7020/12635 [08:10<06:14, 14.98it/s]

In [ ]:
def get_ner_batch(questions):
    # Disable components we don't need to save speed
    with nlp.select_pipes(enable=['tok2vec', 'ner']):
        docs = list(nlp.pipe(questions, batch_size=1000))
    return [set([ent.text for ent in doc.ents]) for doc in docs]

# 1. Get all entities first
q1_ents = get_ner_batch(rawDf['question1'].astype(str))
q2_ents = get_ner_batch(rawDf['question2'].astype(str))

# 2. Compare the sets
def compare_ents(e1, e2):
    if len(e1) == 0 and len(e2) == 0: return 1.0
    return 1.0 if len(e1.intersection(e2)) > 0 else 0.0

rawDf['ner_overlap'] = [compare_ents(a, b) for a, b in zip(q1_ents, q2_ents)]

In [ ]:
def fast_jaccard(df):
    results = []
    # Using zip is significantly faster than .apply(axis=1)
    for q1, q2 in zip(df['question1'], df['question2']):
        s1 = set(str(q1).lower().split())
        s2 = set(str(q2).lower().split())
        inter = len(s1 & s2)
        union = len(s1 | s2)
        results.append(inter / union if union != 0 else 0)
    return results

rawDf['jaccard_sim'] = fast_jaccard(rawDf)

In [ ]:
rawDf.to_csv("../Data/manualTrain.csv",index=False)